# 第四版最终候选：FP32线程调优＋检测/分类流水线

保留第二版FPGA和第三版回退路径，不覆盖旧文件。只在数值、后台采集、延迟及稳定性满足条件时选择新配置。目标是否达到由本轮板测决定，不预设提速结果。

先停止笔记本点餐程序，停止原板端服务器并等待ORDER_BOARD_SERVER_STOPPED，再关闭旧内核。保持摄像头和物理朝向不变。此文件用新的Python 3内核，逐个代码框运行，不运行全部。Notebook允许保存输出，不参与固定文件哈希。


## 1．离线运行库
首次解压到本程序vendor目录；不联网，不修改系统包，不加载硬件。

In [ ]:
from pathlib import Path
import os,sys
ROOT=Path('/home/xilinx/jupyter_notebooks')
APP=ROOT/'gesture_order_accel_v4'
for name in ('board_server','order_core','session_guard','setup_runtime','v3_runtime','v4_runtime','pipeline_runtime','validate_v4','start_v4'):
    module=sys.modules.get(name)
    if module is not None and Path(module.__file__).resolve()!=(APP/(name+'.py')).resolve():
        raise RuntimeError('当前内核缓存了其他版本的 '+name+'，请重启本文件内核再从第1部分运行。')
sys.path.insert(0,str(APP));os.chdir(APP)
from setup_runtime import prepare
prepare(ROOT)


## 2．检查现有模型、FPGA v2、摄像头和端口
正常显示V4_PREFLIGHT_OK，本步不下载比特流。

In [ ]:
from start_v4 import preflight
preflight(ROOT)


## 3．首次完整板测

使用包内48张实拍图及3张合成图，不用摆手，但真实摄像头需保持连接。14组配置逐组校验，再正反顺序测28组；每组2次预热＋12个结果。通常按15—30分钟预留。输出持续显示图像检查/结果计数，不要因为[*]提前停止。

既测检测＋分类耗时，也测每帧来源到结果的延迟，以及相邻结果间隔。流水线可能只提高吞吐，不一定降低每帧延迟，报告会分开标记。CPU路径只对照，不会被选作FPGA演示后端。

等待V4_VALIDATION_OK及SELECTED FPGA CONFIG。若无稳健收益，明确选v3_reference，不假装第四版达标。任何报错先运行第5部分导出，不删除检查。导出时会保存失败前报告。


In [ ]:
from validate_v4 import run
v4_report=None
v4_report=run(ROOT,samples=12)
print('数值/资源检查通过:',v4_report['passed'])
print('选定配置:',v4_report.get('selected_config'))
print('处理耗时相对第二版下降:',v4_report.get('processing_reduction_vs_v2_percent'),'%')
print('处理耗时相对第二版下降25%:',v4_report.get('processing_25percent_vs_v2_met'))
print('每帧延迟相对第三版下降:',v4_report.get('frame_age_reduction_vs_v3_percent'),'%')
print('结果间隔相对第三版下降:',v4_report.get('output_interval_reduction_vs_v3_percent'),'%')
print('每帧延迟相对第三版下降25%:',v4_report.get('latency_25percent_vs_v3_met'))
print('吞吐相对第三版提高25%:',v4_report.get('throughput_25percent_vs_v3_met'))


## 4．真实点餐回归

第3部分完成后运行。等待ORDER_BOARD_SERVER_READY port=8765，再到笔记本运行原start_laptop.bat。板端保持[*]正常。界面/HDMI连接不变。

先三道菜各一份，共33元，提交模拟订单。再验证V返回和取消退出、连续点赞5秒只加一份、移手后握拳减一份、断网期间无动作且重连保留购物车并移手解锁。流水线改变了时序，第四版必须重新验证这些项目，不能直接沿用第三版确认。

如想先核对性能，跳过本部分，直接第5部分导出。


In [ ]:
from start_v4 import run_selected
run_selected(v4_report,ROOT)


## 5．停止后导出

笔记本Ctrl+C → 板端顶部■一次 → 等ORDER_BOARD_SERVER_STOPPED → 运行本框。只测速未启动服务器时可直接导出。若线程未退出，先记录报错，不再加载硬件。

发回下面的板端ZIP。若做了点餐，再把笔记本本次logs压缩发回。只有性能及真实点餐回归都确认后，才将本轮选中配置定为最终版。


In [ ]:
from validate_v4 import export_results
from IPython.display import HTML,display
from urllib.parse import quote
archive=export_results()
href='/files/'+quote(archive.relative_to(ROOT).as_posix(),safe='/')
display(HTML('<a href="'+href+'" download>下载 gesture_order_accel_v4_results.zip</a>'))


## 6．后续启动（首次不要运行）

后续重启内核/板卡后，只需运行第1、2部分，然后本框读取最近一次成功板测报告并启动，不必重复长时间测速。代码、硬件与报告必须匹配；若最后一轮测试失败，会停止而不是悄悄使用更早结果。

停止与导出仍用第5部分。未完成实际点餐回归时，这仍是最终候选。


In [ ]:
from start_v4 import load_last_report,run_selected
v4_report=load_last_report()
run_selected(v4_report,ROOT)
